# Request boundaries — testing the decision logic (Q29)

The participant round records turns, not requests, so a rule has to decide
where one request ends and the next begins. This notebook tests that rule
against a simulated run, where the true boundaries are known: **one execution
is one request**.

**Status: rerun on the B11 regression run (`e097f78e`) alone: 31 cases, 176 executions.**
A copy of `08_segmentation.ipynb` with only the pinned run changed; the
original records the D82 probe (`287e7780`) pooled with `42cab17a`. The
baseline and the regression run are read in separate copies rather than
pooled, because they are two agent builds and a fused run should not mix
them. Sections 1 and 4 describe this run; the predictions in section 1
were stated before the probe and are kept as written.

## The rule under test (`shapes.segment_with`)

For each turn after the first in a Tonic run, the first rule that applies
decides:

| # | signal | reads | decision |
|---|---|---|---|
| 1 | **reference** | the turn's tool arguments against the ids the current request's earlier turns offered or touched (reply place refs, tool-argument ids) | acts on one → **same request** |
| 2 | **slash** | `study_suggestions.agent_intent` | carries a command → **new request** |
| 3 | **shape** | the turn's shape (`shapes7`) | shape changed → new request; no tool → same request |

Reference and slash are **optional evidence**. The agent reaches every path
without a command, so when a slash is absent the rule falls back to the shape.
Reference outranks slash because a member can use a command to continue
("/plan put the second one on Saturday"). The rule is **causal**: a turn is
decided from itself and earlier turns only. The same function therefore
labels a run turn by turn as it happens, or in one pass afterwards, with
identical results.

## What is measured

- **Split** (one request cut in two): boundaries drawn *inside* an execution.
- **Merge** (two requests read as one): executions fused into synthetic runs,
  and the seams a rule misses.
- **Each signal's share:** four variants, from shape only to both signals.
- **Slashes stripped:** the same turns with every command removed, standing
  in for a member who types free text. This is an approximation: the agent
  acted under the command's toolset, and stripping removes the signal, not
  its effect on the agent.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, shapes
from src import segmentation_bench as bench
from src.cases import load_cases

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

# The runs under test: the D82 probe (the two multi-step seeds, n = 3) and the
# 29-case run 42cab17a, pooled so fused runs mix the new executions with the
# old ones. Keys carry the run, the case and the repetition.
RUN_IDS = ["e097f78e-3326-4aea-acaa-0b904d3b0a1c"]  # regression run, 31 cases (rerun of 08 on this run alone; section 4 prose describes 287e7780 + 42cab17a)
MULTI_STEP = ["scheduling_dinner_from_offer", "scheduling_afternoon_then_tram"]

executions, spanning = {}, []
for rid in RUN_IDS:
    print(survey.headline(rid))
    print()
    loaded = survey.turn_signals(rid)
    spanning += loaded.pop(None)
    executions.update({f"{rid[:8]}:{k}": v for k, v in loaded.items()})
present = sorted(k for k in executions if k.split(":")[1].split("#")[0] in MULTI_STEP)
print(f"{len(executions)} executions loaded; spanning more than one Tonic run: {spanning or 'none'}")
print(f"multi-step executions: {len(present)}")
assert not spanning, "an execution is not one request here; the ground truth does not hold"


run e097f78e  started 2026-09-26T11:18:10
  notes         B11 regression run against baseline 1a872c09 — started by tools/overnight_baseline.py --regress
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.8 @ 68062d12787e
  suite         a1d6a7f96d8d
  executions    176  (+0 superseded)
  failed        0
  scored        176, mean 9.48
  cost          $4.4102



176 executions loaded; spanning more than one Tonic run: none
multi-step executions: 16


## 1. Split error — within executions

Every execution is one request, so every value above 1 is a split. Only
multi-step executions can split. For the others the rule's first boundary is
never inside them.

In [2]:
within = pd.DataFrame({v: bench.within(executions, v) for v in bench.VARIANTS})
within_stripped = pd.DataFrame({v: bench.within(bench.strip_slashes(executions), v) for v in bench.VARIANTS})

split = pd.DataFrame({
    "as recorded": (within > 1).sum(),
    "slashes stripped": (within_stripped > 1).sum(),
}).T
print(f"executions cut into more than one request, of {len(within)}:")
split

executions cut into more than one request, of 176:


,shape only,+ slash,+ reference,+ slash + reference
as recorded,16,16,2,2
slashes stripped,16,16,2,2


In [3]:
# The multi-step executions turn by turn: what each signal saw and decided.
rows = []
for case in present:
    for i, t in enumerate(executions[case], start=1):
        rows.append({"case": case, "turn": i, "shape": t.label, "slash": t.intent or "-",
                     "acts on": len(t.acted_on), "offers": len(t.offered)})
    rows.append({"case": case, "turn": "requests", **{v: within.at[case, v] for v in bench.VARIANTS}})
pd.DataFrame(rows).fillna("") if rows else "no multi-step executions in this run"


,case,turn,shape,slash,acts on,offers,shape only,+ slash,+ reference,+ slash + reference
0,e097f78e:scheduling_afternoon_then_tram#1,1,scheduling.draft,plan,5.0,5.0,,,,
1,e097f78e:scheduling_afternoon_then_tram#1,2,other,-,5.0,5.0,,,,
2,e097f78e:scheduling_afternoon_then_tram#1,requests,,,,,2.0,2.0,1.0,1.0
3,e097f78e:scheduling_afternoon_then_tram#2,1,scheduling.draft,plan,5.0,5.0,,,,
4,e097f78e:scheduling_afternoon_then_tram#2,2,other,-,6.0,6.0,,,,
5,e097f78e:scheduling_afternoon_then_tram#2,requests,,,,,2.0,2.0,1.0,1.0
6,e097f78e:scheduling_afternoon_then_tram#3,1,scheduling.draft,plan,5.0,5.0,,,,
7,e097f78e:scheduling_afternoon_then_tram#3,2,other,-,6.0,6.0,,,,
8,e097f78e:scheduling_afternoon_then_tram#3,requests,,,,,2.0,2.0,1.0,1.0
9,e097f78e:scheduling_afternoon_then_tram#4,1,scheduling.draft,plan,2.0,3.0,,,,


**Predictions, stated before the run.**

- `scheduling_dinner_from_offer`: shape alone cuts it in two (venue.search,
  then scheduling.draft). **With reference it is one**, because the draft
  acts on a place the search's reply offered.
- `scheduling_afternoon_then_tram`: cut in two **under every variant**. The
  move acts on the tram, which the draft did not offer: the continuity is in
  the member's words ("the end of that afternoon"), not in any id. If that
  holds, draft → edit is a split no signal in the record can repair, and it is
  reported as a known limit rather than tuned away.

If either prediction fails, the rule or the ids it reads are not what this
notebook assumes, and that needs finding before Q29 is ruled on.

**Result on this run (`e097f78e`, n = 8 per seed).**

- **Dinner: as predicted, every repetition.** Shape alone cuts all 8 in
  two; reference joins all 8, because each draft holds a place the search's
  reply offered.
- **Afternoon: joined in 6 of 8, split in 2, and the split is the
  predicted one.** Where the agent's second turn re-proposed the draft and
  moved the tram together (shape `other`), reference joins the two turns
  through the re-proposed draft's place ids. Where it only moved the tram
  (`scheduling.edit`), the second turn acts on nothing the draft offered and
  every variant splits it. That is the probe's prediction, now observed:
  the link is in the member's words, not in any id.
- **No single-shape case splits in this run.** (In the baseline,
  `factual_qa_lisbon_surf_conditions` split in 3 of 8 through a mixed
  first turn; here none of its repetitions mixed.)
- **Slash does not change any of them.** No execution carries a command
  after its first turn.


## 2. Merge error — fused runs

All ordered pairs, then random runs of 3, 5 and 8 requests (2 000 each, seed
28), each scored under every variant, as recorded and with slashes stripped.
`false boundaries per run` is the split error again, now inside fused runs.

In [4]:
samples = {"all pairs": bench.all_pairs(executions)}
for k in (3, 5, 8):
    samples[f"{k} requests"] = bench.random_runs(executions, k, 2000)

rows = []
for label, source in (("as recorded", executions), ("slashes stripped", bench.strip_slashes(executions))):
    for sample, sequences in samples.items():
        for variant in bench.VARIANTS:
            rows.append({"slashes": label, "sample": sample, "variant": variant,
                         **bench.score(source, variant, sequences)})
fused = pd.DataFrame(rows)
table = fused.pivot_table(index=["slashes", "sample"], columns="variant",
                          values="boundaries found", sort=False)[list(bench.VARIANTS)]
print("boundaries found (1 - merge rate)")
table.map("{:.1%}".format)

boundaries found (1 - merge rate)


variant                     shape only + slash + reference + slash + reference
slashes          sample                                                       
as recorded      all pairs       83.7%   91.9%       83.6%               91.0%
                 3 requests      84.1%   92.0%       83.9%               90.9%
                 5 requests      83.7%   91.9%       83.5%               90.8%
                 8 requests      83.5%   91.7%       83.3%               90.5%
slashes stripped all pairs       83.7%   83.7%       83.6%               83.6%
                 3 requests      84.1%   84.1%       83.9%               83.9%
                 5 requests      83.7%   83.7%       83.5%               83.5%
                 8 requests      83.5%   83.5%       83.3%               83.3%

In [5]:
print("false boundaries per run (split error inside fused runs)")
fused.pivot_table(index=["slashes", "sample"], columns="variant",
                  values="false boundaries per run", sort=False)[list(bench.VARIANTS)].map("{:.3f}".format)

false boundaries per run (split error inside fused runs)


variant                     shape only + slash + reference + slash + reference
slashes          sample                                                       
as recorded      all pairs       0.182   0.182       0.023               0.023
                 3 requests      0.280   0.280       0.030               0.030
                 5 requests      0.458   0.458       0.058               0.058
                 8 requests      0.734   0.734       0.094               0.094
slashes stripped all pairs       0.182   0.182       0.023               0.023
                 3 requests      0.280   0.280       0.030               0.030
                 5 requests      0.458   0.458       0.058               0.058
                 8 requests      0.734   0.734       0.094               0.094

## 3. What reference merges that it should not

Reference joins a turn to the request before it when the turn acts on an id
that request offered. Two *different* requests can share a place: two Belém
mornings both drafting the same bakery, for example. Fused, they read as one.
These are the pairs where adding reference loses a seam that shape and slash
alone would have found.

In [6]:
wrong = []
for a, b in bench.all_pairs(executions):
    base = bench.score(executions, "+ slash", [(a, b)])["runs exact"]
    ref = bench.score(executions, "+ slash + reference", [(a, b)])["runs exact"]
    if base == 1 and ref == 0:
        wrong.append({"first": a, "second": b})
print(f"pairs reference merges wrongly: {len(wrong)} of {len(bench.all_pairs(executions))}")
pd.DataFrame(wrong).head(20) if wrong else "none"


pairs reference merges wrongly: 240 of 30800


,first,second
0,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#2
1,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#3
2,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#4
3,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#5
4,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#6
5,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#7
6,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning#8
7,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning__dir2_on_the...
8,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning__dir2_on_the...
9,e097f78e:scheduling_belem_morning#1,e097f78e:scheduling_belem_morning__dir2_on_the...


## 4. Reading this run

*The B11 regression run (`e097f78e`): 31 cases, 176 executions, 16 of them
multi-step (the two D81 seeds at n = 8). Every execution is exactly one Tonic
run.*

**Split error: 16 of 176 executions are cut in two by shape and slash;
reference leaves 2.** In fused runs that is 0.182 false boundaries per
pair without reference and 0.023 with it (0.734 and 0.094 per run of eight
requests). What reference cannot repair is a follow-up linked only by
wording (the afternoon seed when the agent only moves the tram). Slash
helps with neither.

**Merge error: the D100 rule holds at 31 cases.** Over all 30 800 ordered
pairs, shape alone finds 83.7 % of boundaries, shape + slash (D100) 91.9 %,
shape + reference 83.6 %, and all three 91.0 %. Runs of 3, 5 and 8 requests
stay within half a point of the pairs. Without reference, a boundary can
only be missed where the next request opens with the same shape and no command. With slashes stripped, slash contributes
nothing and the figures fall back to shape alone.

**Reference's false merges: 240 of 30 800 pairs, all between Belém-morning
drafts** (the seed and its directional variants), which share places. It is
the same effect `42cab17a` showed, grown with the repetitions.

**`other` and `advisory` are not empty here.** 11 turns are `other`
(the afternoon seed's draft + move, the packing list's weather + climate, one Sintra Wikipedia + search)
and 20 are `advisory`, all second turns of two-turn executions that called
no tool. `07_shapes` found both empty on the 29-case run `42cab17a`; that
statement holds for that run only.

**What this means for the study.** D100 (shape + slash, no reference) is
confirmed on the full suite at n = 8: about 92 % of boundaries found, and a
split error of about one false boundary per five fused pairs, which
reference would mostly remove at the cost of the Belém merges. The trade
D100 accepted is unchanged in kind; its size is now measured on the runs the
study compares.
